# Automatic SSS matching report

Run the full pipeline (PhysDNet inference + masks + multi-method matching sweep)
for a manually chosen list of timestamp pairs and dump report plots per pair.

Workflow:
1. Edit `TIMESTAMP_PAIRS` and (optionally) `SWEEP_SPECS` below.
2. Run every cell. Each pair lands in
   `output_simplified/report/{ts1}_{ts2}/`.


In [ ]:
import os
import sys
import time
from pathlib import Path

_REPO_ROOT = Path.cwd().parent if Path.cwd().name == "hands_on_utils" else Path.cwd()
if str(_REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(_REPO_ROOT))

from typing import Optional

import cv2
import numpy as np
import torch
import matplotlib.pyplot as plt

from prep import *
from inference import *
from xtf_utils import *
from masks import *
from minima_pipeline import *

from report_utils import (
    SweepSpec, run_sweep, make_report_dir, select_top_runs,
    plot_filtering_funnel, plot_utm_residuals,
    plot_homography_params, plot_warp_diff, plot_summary_table,
    generate_cross_pair_report,
)

torch.set_grad_enabled(False)


In [2]:
# ---------------------------------------------------------------------------
# Paths and constants
# ---------------------------------------------------------------------------
xtf_file = r"2025-09-24_09-25-24_0.xtf"
xtf_dir = r"/home/max/HoP-SSS-Loop-Closure/data"
output_dir = r"/home/max/HoP-SSS-Loop-Closure/output_simplified"
weight_path = r"/home/max/HoP-SSS-Loop-Closure/best_model_v610_jaguar.pth"

LIGHTGLUE_REPO = r""
if LIGHTGLUE_REPO and LIGHTGLUE_REPO not in sys.path:
    sys.path.insert(0, LIGHTGLUE_REPO)

MINIMA_CKPT = r"/home/max/HoP-SSS-Loop-Closure/MINIMA/weights/minima_lightglue.pth"

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# XTF section extraction
segment_size = 2000
upper_limit = 2 ** 15

# Mask / matching settings
TERRAIN_MASK_K = 1.0
MASK_COMPONENTS = ["shadow", "blind"]
FLIP_LEFT_FINAL_MASK = False
MATCH_MODE = "both"
FLIP_LEFT_INPUT_IMAGE = False
APPLY_MASK = True
APPLY_UTM_DISTANCE_FILTER = True
UTM_DISTANCE_THRESHOLD_METERS = 1.0

# Matcher defaults (used by SweepSpec entries that omit a kwarg)
LIGHTGLUE_MAX_NUM_KEYPOINTS = 4096
LIGHTGLUE_RESIZE = None
LIGHTGLUE_DEPTH_CONFIDENCE = 0.95
LIGHTGLUE_WIDTH_CONFIDENCE = 0.99
LIGHTGLUE_FILTER_THRESHOLD = 0.1
LIGHTGLUE_MP = False
LIGHTGLUE_FLASH = True
LIGHTGLUE_COMPILE = False

SIFT_NFEATURES = 8000
SIFT_N_OCTAVE_LAYERS = 3
SIFT_CONTRAST_THRESHOLD = 0.01
SIFT_EDGE_THRESHOLD = 10
SIFT_SIGMA = 1.6
SIFT_MATCHER = "flann"
SIFT_RATIO_TEST = 0.75
SIFT_CROSS_CHECK = False
SIFT_FLANN_TREES = 5
SIFT_FLANN_CHECKS = 64
SIFT_ENFORCE_UNIQUE_MATCHES = True
SIFT_MAX_MATCHES = None

# RANSAC
RANSAC_REPROJ_THRESHOLD_PIXEL = 10.0
RANSAC_REPROJ_THRESHOLD_UTM = 2.0
RANSAC_MAX_ITERS = 50_000
RANSAC_CONFIDENCE = 0.995
RANSAC_OUTLIER_PERCENT = 0.6


Using device: cuda:0


In [ ]:
# ---------------------------------------------------------------------------
# >>> EDIT THIS CELL <<<
# Timestamp pairs to process. Each entry is (ts1, ts2) or (ts1, ts2, label).
# The optional label is used in the report folder name AND as the x-tick on
# the cross-pair plots, so pick something short and descriptive
# (e.g. "easy_parallel"). Use underscores rather than spaces.
# The order of this list is preserved on the cross-pair plots (left -> right),
# so put pairs in difficulty order.
# ---------------------------------------------------------------------------
TIMESTAMP_PAIRS = [
    (4750,  1250,  "easy_close_parallel"),
    (48200, 46150, "hard_close_anti"),
    (13700, 11300, "medium_close"),
    (29800, 68100, "cross_overlap"),
    (7650,  1250,  "easy_far_parallel"),
    (52500, 46150, "hard_far_anti"),
    (29800, 68350, "cross_far_overlap"),
]

# 14-spec sweep (3 methods x both image sources x small param grid).
SWEEP_SPECS: list[SweepSpec] = []
for src in ("raw", "rho_gray"):
    s = "raw" if src == "raw" else "rho"

    # SIFT: vary Lowe ratio
    for ratio in (0.7, 0.8):
        SWEEP_SPECS.append(SweepSpec(
            f"sift_r{ratio}_{s}", "sift",
            {"ratio_test": ratio, "nfeatures": 8000},
            src, "SIFT", f"ratio={ratio}",
        ))

    # LightGlue+SuperPoint: vary filter_threshold
    for ft in (0.05, 0.1, 0.2):
        SWEEP_SPECS.append(SweepSpec(
            f"lgsp_ft{ft}_{s}", "lightglue_superpoint",
            {"filter_threshold": ft, "max_num_keypoints": 4096},
            src, "LightGlue+SuperPoint", f"ft={ft}",
        ))

    # MINIMA (sp_lg): single config
    SWEEP_SPECS.append(SweepSpec(
        f"minima_{s}", "minima_sp_lg", {},
        src, "MINIMA(sp_lg)", "default",
    ))

print(f"Pairs to process: {len(TIMESTAMP_PAIRS)}")
print(f"Specs per pair:   {len(SWEEP_SPECS)}")


In [4]:
# Load the XTF file once and detect the straight-line groups (turn segments
# are dropped). Both are reused across every pair.
data_load = load_xtf(xtf_dir + r"/" + xtf_file)
swaths, trajectory, altitude, roll, pitch, yaw = calculate_swath_positions(data_load)

_filter, _dy = lowpass_and_derivative_5point(
    np.arange(len(np.unwrap(yaw) * 180 / np.pi)),
    np.unwrap(yaw) * 180 / np.pi,
    0.01,
)

# Threshold the yaw derivative to drop turning pings.
turn_threshold = 0.08
turn_idx = np.where(np.abs(_dy) > turn_threshold)[0]
straight_idx = np.delete(np.arange(swaths.shape[0]), turn_idx)

# Split into contiguous groups and keep large ones.
group_split_jumps = np.where(np.diff(straight_idx) > 1)[0] + 1
groups = np.split(straight_idx, group_split_jumps)
min_group_size = 29
groups_indices = [g for g in groups if len(g) > min_group_size]

print(f"Loaded {len(data_load)} pings, {len(groups_indices)} straight-line groups")


---------------------
FileFormat: 123
SystemType: 1
RecordingProgramName: b'IQUA'
RecordingProgramVersion: b'100'
SonarName: b'ScoutMkII_ros'
SonarType: 44
NoteString: b'Marine Sonic Technology Corp.'
ThisFileName: b'/home/user/logs/mk_ii/2025-09-24_09-25-24_0.xtf'
NavUnits: 3
NumberOfSonarChannels: 2
NumberOfBathymetryChannels: 0
NumberOfSnippetChannels: 0
NumberOfForwardLookArrays: 0
NumberOfEchoStrengthChannels: 0
NumberOfInterferometryChannels: 0
Reserved1: 0
Reserved2: 0
ReferencePointHeight: 0.0
ProjectionType: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
SpheriodType: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
NavigationLatency: 0
OriginY: 0.0
OriginX: 0.0
NavOffsetY: 0.0
NavOffsetX: 0.0
NavOffsetZ: 0.0
NavOffsetYaw: 0.0
MRUOffsetY: 0.0
MRUOffsetX: 0.0
MRUOffsetZ: 0.0
MRUOffsetYaw: 0.0
MRUOffsetPitch: 0.0
MRUOffsetRoll: 0.0
ChanInfo: [<pyxtf.xtf_ctypes.XTFChanInfo object at 0x71c2589575d0>, <pyxtf.xtf_ctypes.XTFChanInfo object at 0x71c258957650>, <pyxtf.xtf_ctypes.XTFChanInfo object at 0x71c2589576

## Matcher classes

Copied verbatim from the main pipeline notebook (cell 29) so this notebook is
self-contained. `build_matcher` is replaced by the spec-aware
`make_matcher_for_spec` defined further down.


In [5]:
def _ensure_gray_uint8_for_matching(img: np.ndarray) -> np.ndarray:
    img = ensure_uint8_image(img)

    if img.ndim == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    return np.ascontiguousarray(img)


class LightGlueFeatureMatcher:
    """
    Direct LightGlue matcher using the cloned cvg/LightGlue repository.

    feature_type:
        "superpoint" -> SuperPoint + LightGlue
        "sift"       -> SIFT + LightGlue
    """

    def __init__(
        self,
        feature_type: str = "superpoint",
        device: Optional[torch.device] = None,
        max_num_keypoints: Optional[int] = 4096,
        resize=None,
        depth_confidence: float = 0.95,
        width_confidence: float = 0.99,
        filter_threshold: float = 0.1,
        mp: bool = False,
        flash: bool = True,
        compile_matcher: bool = False,
    ) -> None:
        from lightglue import LightGlue, SuperPoint, SIFT
        from lightglue.utils import rbd, numpy_image_to_torch

        self.feature_type = feature_type.lower().strip()
        self.device = device or torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.resize = resize
        self.rbd = rbd
        self.numpy_image_to_torch = numpy_image_to_torch

        if self.feature_type == "superpoint":
            self.extractor = SuperPoint(max_num_keypoints=max_num_keypoints).eval().to(self.device)
            lg_features = "superpoint"
        elif self.feature_type == "sift":
            self.extractor = SIFT(max_num_keypoints=max_num_keypoints).eval().to(self.device)
            lg_features = "sift"
        else:
            raise ValueError("feature_type must be 'superpoint' or 'sift'")

        self.matcher = LightGlue(
            features=lg_features,
            depth_confidence=depth_confidence,
            width_confidence=width_confidence,
            filter_threshold=filter_threshold,
            mp=mp,
            flash=flash,
        ).eval().to(self.device)

        if compile_matcher and hasattr(self.matcher, "compile"):
            self.matcher.compile(mode="reduce-overhead")

    def _to_lightglue_image(self, img: np.ndarray):
        img = _ensure_gray_uint8_for_matching(img)
        image = self.numpy_image_to_torch(img).to(self.device)
        return image

    def match_images(self, img0: np.ndarray, img1: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        image0 = self._to_lightglue_image(img0)
        image1 = self._to_lightglue_image(img1)

        with torch.inference_mode():
            feats0 = self.extractor.extract(image0, resize=self.resize)
            feats1 = self.extractor.extract(image1, resize=self.resize)
            matches01 = self.matcher({"image0": feats0, "image1": feats1})

        feats0, feats1, matches01 = [self.rbd(x) for x in [feats0, feats1, matches01]]

        matches = matches01["matches"]
        if matches.numel() == 0:
            return (
                np.empty((0, 2), dtype=np.float32),
                np.empty((0, 2), dtype=np.float32),
                np.empty((0,), dtype=np.float32),
            )

        mkpts0 = feats0["keypoints"][matches[..., 0]]
        mkpts1 = feats1["keypoints"][matches[..., 1]]

        if "scores" in matches01:
            mconf = matches01["scores"]
        elif "matching_scores0" in matches01:
            mconf = matches01["matching_scores0"][matches[..., 0]]
        else:
            mconf = torch.ones(matches.shape[0], device=matches.device, dtype=torch.float32)

        return (
            mkpts0.detach().cpu().numpy().astype(np.float32),
            mkpts1.detach().cpu().numpy().astype(np.float32),
            mconf.detach().cpu().numpy().reshape(-1).astype(np.float32),
        )


class SIFTMatcher:
    """
    Pure OpenCV SIFT + descriptor matching.

    This backend does not use LightGlue. It is useful as a classical baseline.
    """

    def __init__(
        self,
        nfeatures: int = 8000,
        n_octave_layers: int = 3,
        contrast_threshold: float = 0.01,
        edge_threshold: float = 10,
        sigma: float = 1.6,
        matcher_type: str = "flann",
        ratio_test: float = 0.75,
        cross_check: bool = False,
        flann_trees: int = 5,
        flann_checks: int = 64,
        enforce_unique_matches: bool = True,
        max_matches: Optional[int] = None,
    ) -> None:
        self.sift = cv2.SIFT_create(
            nfeatures=nfeatures,
            nOctaveLayers=n_octave_layers,
            contrastThreshold=contrast_threshold,
            edgeThreshold=edge_threshold,
            sigma=sigma,
        )
        self.matcher_type = matcher_type.lower().strip()
        self.ratio_test = ratio_test
        self.cross_check = cross_check
        self.flann_trees = flann_trees
        self.flann_checks = flann_checks
        self.enforce_unique_matches = enforce_unique_matches
        self.max_matches = max_matches

    @staticmethod
    def _unique_matches_by_distance(matches: list[cv2.DMatch]) -> list[cv2.DMatch]:
        # First keep the best match per query keypoint.
        best_by_query = {}
        for m in matches:
            old = best_by_query.get(m.queryIdx)
            if old is None or m.distance < old.distance:
                best_by_query[m.queryIdx] = m

        # Then keep the best match per train keypoint.
        best_by_train = {}
        for m in best_by_query.values():
            old = best_by_train.get(m.trainIdx)
            if old is None or m.distance < old.distance:
                best_by_train[m.trainIdx] = m

        return list(best_by_train.values())

    def _match_descriptors(self, desc0: np.ndarray, desc1: np.ndarray) -> list[cv2.DMatch]:
        if self.matcher_type == "bf":
            bf = cv2.BFMatcher(cv2.NORM_L2, crossCheck=self.cross_check)

            if self.cross_check:
                matches = bf.match(desc0, desc1)
            else:
                raw_matches = bf.knnMatch(desc0, desc1, k=2)
                matches = [m for m, n in raw_matches if m.distance < self.ratio_test * n.distance]

        elif self.matcher_type == "flann":
            index_params = dict(algorithm=1, trees=self.flann_trees)  # 1 = KDTree
            search_params = dict(checks=self.flann_checks)
            flann = cv2.FlannBasedMatcher(index_params, search_params)

            raw_matches = flann.knnMatch(desc0.astype(np.float32), desc1.astype(np.float32), k=2)
            matches = []
            for pair in raw_matches:
                if len(pair) < 2:
                    continue
                m, n = pair
                if m.distance < self.ratio_test * n.distance:
                    matches.append(m)
        else:
            raise ValueError("SIFT_MATCHER must be 'flann' or 'bf'")

        if self.enforce_unique_matches:
            matches = self._unique_matches_by_distance(matches)

        matches = sorted(matches, key=lambda m: m.distance)

        if self.max_matches is not None:
            matches = matches[: int(self.max_matches)]

        return matches

    def match_images(self, img0: np.ndarray, img1: np.ndarray) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        gray0 = _ensure_gray_uint8_for_matching(img0)
        gray1 = _ensure_gray_uint8_for_matching(img1)

        kpts0, desc0 = self.sift.detectAndCompute(gray0, None)
        kpts1, desc1 = self.sift.detectAndCompute(gray1, None)

        if desc0 is None or desc1 is None or len(kpts0) == 0 or len(kpts1) == 0:
            return (
                np.empty((0, 2), dtype=np.float32),
                np.empty((0, 2), dtype=np.float32),
                np.empty((0,), dtype=np.float32),
            )

        matches = self._match_descriptors(desc0, desc1)

        if len(matches) == 0:
            return (
                np.empty((0, 2), dtype=np.float32),
                np.empty((0, 2), dtype=np.float32),
                np.empty((0,), dtype=np.float32),
            )

        mkpts0 = np.array([kpts0[m.queryIdx].pt for m in matches], dtype=np.float32)
        mkpts1 = np.array([kpts1[m.trainIdx].pt for m in matches], dtype=np.float32)

        distances = np.array([m.distance for m in matches], dtype=np.float32)
        mconf = 1.0 / (1.0 + distances)

        return mkpts0, mkpts1, mconf


## Pipeline helpers

Copied verbatim from the main pipeline notebook (cell 24). These override the
imported `minima_pipeline.*` helpers with the notebook's mask / orientation
handling and the UTM-distance filter.


In [6]:
# `display_figure` is referenced by `build_final_masks`. The main notebook
# defines it in cell 5; here we make it a no-op so the batch loop doesn't spam
# interactive figures (the masks themselves are saved to disk by build_final_masks
# via cv2.imwrite). Replace with the plotting version below if you want to
# inspect masks per pair.
def display_figure(images_dict: dict, lower: int, upper: int, main_title: str):
    return None

# Plotting version (uncomment to enable):
# def display_figure(images_dict: dict, lower: int, upper: int, main_title: str):
#     images = list(images_dict.values())
#     titles = list(images_dict.keys())
#     n = len(images)
#     fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
#     if n == 1:
#         axes = [axes]
#     for ax, img, title in zip(axes, images, titles):
#         ax.imshow(img, cmap="gray"); ax.axis("off"); ax.set_title(title)
#     plt.suptitle(f"{main_title}: {lower} to {upper}")
#     plt.tight_layout(); plt.show()


def combine_masks_for_matching(
    masks: list[np.ndarray],
    side: str,
    flip_left_final: bool = False,
) -> np.ndarray:
    """
    Build the exact final rejection mask used by the matcher.

    Mask convention:
        0     -> valid / keep
        > 0   -> invalid / reject

    The safe rule is:
        1. Keep all selected mask components in the orientation produced by the
           preprocessing/inference code.
        2. Merge them into one final rejection mask.
        3. Optionally flip only the final LEFT mask, controlled by
           FLIP_LEFT_FINAL_MASK.
    """
    if not masks:
        raise ValueError("masks list is empty")

    side = side.lower().strip()
    if side not in {"left", "right"}:
        raise ValueError(f"Invalid side='{side}'. Use 'left' or 'right'.")

    target_shape = np.asarray(masks[0]).shape[:2]
    normalised = []

    for mask in masks:
        mask = np.asarray(mask)

        if mask.ndim == 3:
            mask = cv2.cvtColor(mask, cv2.COLOR_BGR2GRAY)

        if mask.shape[:2] != target_shape:
            mask = cv2.resize(
                mask,
                (target_shape[1], target_shape[0]),
                interpolation=cv2.INTER_NEAREST,
            )

        normalised.append(mask)

    final_mask = (sum(mask > 0 for mask in normalised) > 0).astype(np.uint8) * 255

    if flip_left_final and side == "left":
        final_mask = np.fliplr(final_mask)

    return np.ascontiguousarray(final_mask)


def show_mask_component_debug(section, terrain, side: str, selected_components: dict, final_mask: np.ndarray, title: str):
    """
    Shows the selected mask components and the final mask that will be stored in
    terrain.final_mask.
    """
    n_components = len(selected_components)
    fig, axes = plt.subplots(1, n_components + 1, figsize=(5 * (n_components + 1), 5))

    if n_components + 1 == 1:
        axes = [axes]

    for ax, (name, mask) in zip(axes[:-1], selected_components.items()):
        ax.imshow(mask, cmap="gray")
        ax.set_title(f"{side} component: {name}")
        ax.axis("off")

    axes[-1].imshow(final_mask, cmap="gray")
    flip_state = "flipped" if (side == "left" and FLIP_LEFT_FINAL_MASK) else "not flipped"
    axes[-1].set_title(f"{side} FINAL applied mask ({flip_state})")
    axes[-1].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()



def build_final_masks(section, terrain, output_mask_dir: str, lower: int, upper: int, title: str):
    """
    Builds terrain.final_mask from the selected mask components.

    Mask convention:
        0     -> valid / keep
        > 0   -> invalid / reject

    Orientation rule:
        - shadow, z/height, and blind masks are merged in their native orientation.
        - the final LEFT mask is flipped only if FLIP_LEFT_FINAL_MASK=True.
        - the image flip is controlled separately by FLIP_LEFT_INPUT_IMAGE.
    """
    os.makedirs(output_mask_dir, exist_ok=True)

    # Avoid stale masks from previous notebook executions.
    terrain.final_mask = {}

    for side in ["left", "right"]:
        shadow_key = get_key_by_side(section.shadows, side)
        z_key = get_key_by_side(terrain.z_mask, side)
        blind_key = get_key_by_side(section.blind, side)

        components_by_name = {
            "shadow": section.shadows[shadow_key],
            "z": terrain.z_mask[z_key],
            "blind": section.blind[blind_key],
        }

        selected_components = {}
        selected_masks = []

        for name in MASK_COMPONENTS:
            if name not in components_by_name:
                raise ValueError(
                    f"Invalid mask component '{name}'. "
                    f"Use one of: {list(components_by_name.keys())}"
                )

            selected_components[name] = components_by_name[name]
            selected_masks.append(components_by_name[name])

        final_key = f"{side}_final_mask"
        terrain.final_mask[final_key] = combine_masks_for_matching(
            selected_masks,
            side=side,
            flip_left_final=FLIP_LEFT_FINAL_MASK,
        )

        cv2.imwrite(
            os.path.join(output_mask_dir, f"{final_key}.png"),
            terrain.final_mask[final_key],
        )

        show_mask_component_debug(
            section=section,
            terrain=terrain,
            side=side,
            selected_components=selected_components,
            final_mask=terrain.final_mask[final_key],
            title=f"{title} - {side} mask components and final applied mask",
        )

    display_figure(terrain.final_mask, lower, upper, title)


def prepare_side_image_and_mask(
    inference_obj,
    terrain_obj,
    side: str,
    image_source: str = "rho_gray",
    section_obj=None,
    flip_left_image: bool | None = None,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Returns one side image and its corresponding final mask in matching orientation.

    This local definition intentionally overrides the imported minima_pipeline version
    so notebook changes are not affected by stale imports.

    Orientation rule:
        - LEFT image is flipped when FLIP_LEFT_INPUT_IMAGE=True.
        - Mask orientation is controlled only by FLIP_LEFT_FINAL_MASK during
          build_final_masks(); it is not flipped again here.
    """
    if flip_left_image is None:
        flip_left_image = FLIP_LEFT_INPUT_IMAGE

    side = side.lower().strip()
    if side not in {"left", "right"}:
        raise ValueError(f"Invalid side='{side}'. Use 'left' or 'right'.")

    image_dict = get_image_dict_from_source(
        image_source=image_source,
        inference_obj=inference_obj,
        section_obj=section_obj,
    )

    img_key = get_key_by_side(image_dict, side)
    mask_key = get_key_by_side(terrain_obj.final_mask, side)

    img = image_dict[img_key]
    mask = terrain_obj.final_mask[mask_key]

    if side == "left" and flip_left_image:
        img = np.fliplr(img)

    img = ensure_uint8_image(img)
    img = np.ascontiguousarray(img)

    mask = resize_mask_to_image(mask, img)
    mask = np.ascontiguousarray(mask)

    return img, mask


def build_matching_inputs(
    mode: str,
    inference0,
    inference1,
    terrain0,
    terrain1,
    image_source: str = "rho_gray",
    section0=None,
    section1=None,
    flip_left_image: bool | None = None,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, str, str]:
    """
    Builds the exact images and masks used for matching.

    For MATCH_MODE="both", the convention is always:
        image = [LEFT_in_matching_orientation | RIGHT_native]
        mask  = [LEFT_final_mask             | RIGHT_final_mask]
    """
    if flip_left_image is None:
        flip_left_image = FLIP_LEFT_INPUT_IMAGE

    mode = mode.lower().strip()

    if mode == "both":
        left0_img, left0_mask = prepare_side_image_and_mask(
            inference0,
            terrain0,
            "left",
            image_source=image_source,
            section_obj=section0,
            flip_left_image=flip_left_image,
        )
        right0_img, right0_mask = prepare_side_image_and_mask(
            inference0,
            terrain0,
            "right",
            image_source=image_source,
            section_obj=section0,
            flip_left_image=flip_left_image,
        )

        left1_img, left1_mask = prepare_side_image_and_mask(
            inference1,
            terrain1,
            "left",
            image_source=image_source,
            section_obj=section1,
            flip_left_image=flip_left_image,
        )
        right1_img, right1_mask = prepare_side_image_and_mask(
            inference1,
            terrain1,
            "right",
            image_source=image_source,
            section_obj=section1,
            flip_left_image=flip_left_image,
        )

        img0 = np.ascontiguousarray(np.hstack([left0_img, right0_img]))
        img1 = np.ascontiguousarray(np.hstack([left1_img, right1_img]))

        mask0 = np.ascontiguousarray(np.hstack([left0_mask, right0_mask]))
        mask1 = np.ascontiguousarray(np.hstack([left1_mask, right1_mask]))

        return img0, img1, mask0, mask1, "both", "both"

    valid_modes = {"left-left", "right-right", "left-right", "right-left"}
    if mode not in valid_modes:
        raise ValueError(f"Invalid MATCH_MODE='{mode}'. Use one of: {sorted(valid_modes | {'both'})}")

    side0, side1 = mode.split("-")

    img0, mask0 = prepare_side_image_and_mask(
        inference0,
        terrain0,
        side0,
        image_source=image_source,
        section_obj=section0,
        flip_left_image=flip_left_image,
    )

    img1, mask1 = prepare_side_image_and_mask(
        inference1,
        terrain1,
        side1,
        image_source=image_source,
        section_obj=section1,
        flip_left_image=flip_left_image,
    )

    return img0, img1, mask0, mask1, side0, side1


def get_swaths_for_side(pings: list[pyxtf.XTFPingHeader], side: str) -> np.ndarray:
    """
    Returns the swath grid in the same horizontal orientation as the image used
    for matching.

    If FLIP_LEFT_INPUT_IMAGE=True, left swaths are flipped too. This keeps
    pixel-to-UTM conversion consistent with flipped left image coordinates.
    """
    swaths = extract_swaths_from_calculate_output(calculate_swath_positions(pings))

    side = side.lower().strip()
    mid = swaths.shape[1] // 2

    left_swath = swaths[:, :mid, :]
    right_swath = swaths[:, mid:, :]

    if FLIP_LEFT_INPUT_IMAGE:
        left_swath = np.fliplr(left_swath)

    left_swath = np.ascontiguousarray(left_swath)
    right_swath = np.ascontiguousarray(right_swath)

    if side == "left":
        return left_swath

    if side == "right":
        return right_swath

    if side == "both":
        return np.ascontiguousarray(np.hstack([left_swath, right_swath]))

    raise ValueError(f"Invalid swath side: {side}")


def mask_overlay_for_debug_local(img: np.ndarray, mask: np.ndarray, alpha: float = 0.45) -> np.ndarray:
    """
    White overlay = rejected pixels. Black/no overlay = pixels available for matching.
    """
    img_u8 = ensure_uint8_image(img)
    mask_u8 = ensure_gray(mask)

    if img_u8.ndim == 2:
        base = cv2.cvtColor(img_u8, cv2.COLOR_GRAY2BGR)
    else:
        base = img_u8.copy()

    overlay = base.copy()
    overlay[mask_u8 > 0] = 255
    return cv2.addWeighted(base, 1.0 - alpha, overlay, alpha, 0.0)


def show_matching_inputs_debug(
    mode: str,
    inference0,
    inference1,
    terrain0,
    terrain1,
    image_source: str = "rho_gray",
    section0=None,
    section1=None,
    flip_left_image: bool | None = None,
):
    """
    Displays the exact image/mask arrays that will be passed to the matcher.
    """
    if flip_left_image is None:
        flip_left_image = FLIP_LEFT_INPUT_IMAGE

    img0, img1, mask0, mask1, swath_side0, swath_side1 = build_matching_inputs(
        mode=mode,
        inference0=inference0,
        inference1=inference1,
        terrain0=terrain0,
        terrain1=terrain1,
        image_source=image_source,
        section0=section0,
        section1=section1,
        flip_left_image=flip_left_image,
    )

    print(f"Matching debug mode: {mode}")
    print(f"Image source: {image_source}")
    print(f"FLIP_LEFT_INPUT_IMAGE = {flip_left_image}")
    print(f"FLIP_LEFT_FINAL_MASK  = {FLIP_LEFT_FINAL_MASK}")
    print(f"Image 0 shape: {img0.shape} | Mask 0 shape: {mask0.shape} | Swath side: {swath_side0}")
    print(f"Image 1 shape: {img1.shape} | Mask 1 shape: {mask1.shape} | Swath side: {swath_side1}")
    print("Mask convention: black=keep, white=reject")

    overlay0 = mask_overlay_for_debug_local(img0, mask0)
    overlay1 = mask_overlay_for_debug_local(img1, mask1)

    fig, axes = plt.subplots(2, 3, figsize=(18, 10))

    axes[0, 0].imshow(img0, cmap="gray")
    axes[0, 0].set_title("Image 0 passed to matcher")
    axes[0, 0].axis("off")

    axes[0, 1].imshow(mask0, cmap="gray")
    axes[0, 1].set_title("Final mask 0 applied after matching")
    axes[0, 1].axis("off")

    axes[0, 2].imshow(cv2.cvtColor(overlay0, cv2.COLOR_BGR2RGB))
    axes[0, 2].set_title("Mask 0 overlay on image 0")
    axes[0, 2].axis("off")

    axes[1, 0].imshow(img1, cmap="gray")
    axes[1, 0].set_title("Image 1 passed to matcher")
    axes[1, 0].axis("off")

    axes[1, 1].imshow(mask1, cmap="gray")
    axes[1, 1].set_title("Final mask 1 applied after matching")
    axes[1, 1].axis("off")

    axes[1, 2].imshow(cv2.cvtColor(overlay1, cv2.COLOR_BGR2RGB))
    axes[1, 2].set_title("Mask 1 overlay on image 1")
    axes[1, 2].axis("off")

    plt.suptitle("Exact matching inputs and final masks")
    plt.tight_layout()
    plt.show()

    return img0, img1, mask0, mask1


def filter_utm_matches_by_distance(
    mkpts0_utm: np.ndarray,
    mkpts1_utm: np.ndarray,
    max_distance: Optional[float],
    enabled: bool = False,
) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
    """
    Optionally rejects UTM matches whose paired global coordinates are too far apart.

    If enabled=False, no match is removed and the keep mask is all True.
    Distance units are the same as the UTM coordinates, normally meters.
    """
    mkpts0_utm = np.asarray(mkpts0_utm, dtype=np.float64)
    mkpts1_utm = np.asarray(mkpts1_utm, dtype=np.float64)

    if len(mkpts0_utm) != len(mkpts1_utm):
        raise ValueError("mkpts0_utm and mkpts1_utm must have the same length")

    if len(mkpts0_utm) == 0:
        distances = np.empty((0,), dtype=np.float64)
        keep_mask = np.empty((0,), dtype=bool)
        return mkpts0_utm, mkpts1_utm, distances, keep_mask

    distances = np.linalg.norm(mkpts1_utm - mkpts0_utm, axis=1)

    if not enabled:
        keep_mask = np.ones(len(distances), dtype=bool)
        return mkpts0_utm, mkpts1_utm, distances, keep_mask

    if max_distance is None:
        raise ValueError("UTM_DISTANCE_THRESHOLD_METERS must be set when APPLY_UTM_DISTANCE_FILTER=True")

    max_distance = float(max_distance)
    if max_distance <= 0.0:
        raise ValueError("UTM_DISTANCE_THRESHOLD_METERS must be positive when APPLY_UTM_DISTANCE_FILTER=True")

    keep_mask = distances <= max_distance
    return mkpts0_utm[keep_mask], mkpts1_utm[keep_mask], distances, keep_mask


def run_parametrized_minima_registration(
    matcher,
    mode: str,
    inference0,
    inference1,
    terrain0,
    terrain1,
    pings0: list[pyxtf.XTFPingHeader],
    pings1: list[pyxtf.XTFPingHeader],
    image_source: str = "rho_gray",
    section0=None,
    section1=None,
    apply_mask: bool = True,
    ransac_reproj_threshold_pixel: float = 10.0,
    ransac_reproj_threshold_utm: float = 1.0,
    ransac_max_iters: int = 10_000,
    ransac_confidence: float = 0.995,
    ransac_outlier_percent: float = 0.5,
    flip_left_image: bool | None = None,
    apply_utm_distance_filter: bool = False,
    utm_distance_threshold: Optional[float] = None,
):
    """
    Local notebook registration runner that uses the same image/mask orientation
    as show_matching_inputs_debug().
    """
    if flip_left_image is None:
        flip_left_image = FLIP_LEFT_INPUT_IMAGE

    if pings0 is None or pings1 is None:
        raise ValueError("pings0 and pings1 must always be passed for UTM conversion")

    image_source = image_source.lower().strip()

    img0, img1, mask0, mask1, swath_side0, swath_side1 = build_matching_inputs(
        mode=mode,
        inference0=inference0,
        inference1=inference1,
        terrain0=terrain0,
        terrain1=terrain1,
        image_source=image_source,
        section0=section0,
        section1=section1,
        flip_left_image=flip_left_image,
    )

    swaths0 = get_swaths_for_side(pings0, swath_side0)
    swaths1 = get_swaths_for_side(pings1, swath_side1)

    print(f"Matching mode: {mode}")
    print(f"Image source: {image_source}")
    print(f"Apply mask: {apply_mask}")
    print(f"FLIP_LEFT_INPUT_IMAGE = {flip_left_image}")
    print(f"FLIP_LEFT_FINAL_MASK  = {FLIP_LEFT_FINAL_MASK}")
    print(f"APPLY_UTM_DISTANCE_FILTER = {apply_utm_distance_filter}")
    print(f"UTM_DISTANCE_THRESHOLD_METERS = {utm_distance_threshold}")
    print(f"Image 0 shape: {img0.shape} | swath side: {swath_side0} | swath grid: {swaths0.shape}")
    print(f"Image 1 shape: {img1.shape} | swath side: {swath_side1} | swath grid: {swaths1.shape}")
    print(f"Mask 0 shape:  {mask0.shape}")
    print(f"Mask 1 shape:  {mask1.shape}")

    mkpts0, mkpts1, mconf = matcher.match_images(img0, img1)

    if apply_mask:
        mkpts0_kept, mkpts1_kept, mconf_kept, keep_mask = filter_matches_with_masks(
            mkpts0,
            mkpts1,
            mconf,
            mask0=mask0,
            mask1=mask1,
        )
    else:
        keep_mask = np.ones(len(mkpts0), dtype=bool)
        mkpts0_kept = mkpts0
        mkpts1_kept = mkpts1
        mconf_kept = mconf

    H_pixel, ransac_inliers_pixel = estimate_homography_ransac_custom(
        mkpts0_kept,
        mkpts1_kept,
        ransac_reproj_threshold=ransac_reproj_threshold_pixel,
        max_iters=ransac_max_iters,
        confidence=ransac_confidence,
        model="Euclidean",
        outlier_percent=ransac_outlier_percent,
    )

    mkpts0_utm = pixels_to_utm(mkpts0_kept, swaths0, img0.shape)
    mkpts1_utm = pixels_to_utm(mkpts1_kept, swaths1, img1.shape)

    mkpts0_utm_ransac, mkpts1_utm_ransac, utm_match_distances, utm_distance_keep_mask = filter_utm_matches_by_distance(
        mkpts0_utm,
        mkpts1_utm,
        max_distance=utm_distance_threshold,
        enabled=apply_utm_distance_filter,
    )

    num_utm_distance_rejected = int(len(mkpts0_utm) - len(mkpts0_utm_ransac))
    print(f"After mask filtering: {len(mkpts0_kept)} matches")
    print(f"After UTM distance filter: {len(mkpts0_utm_ransac)} / {len(mkpts0_utm)} matches")
    if apply_utm_distance_filter:
        print(f"Rejected by UTM distance: {num_utm_distance_rejected}")

    H_utm, H_utm_local, mkpts0_utm_local, mkpts1_utm_local, utm_offset0, utm_offset1, ransac_inliers_utm_filtered = estimate_utm_homography_ransac(
        mkpts0_utm_ransac,
        mkpts1_utm_ransac,
        ransac_reproj_threshold=ransac_reproj_threshold_utm,
        max_iters=ransac_max_iters,
        confidence=ransac_confidence,
        outlier_percent=ransac_outlier_percent,
    )

    # Expand back to the post-mask length so result.ransac_inliers aligns with
    # result.mkpts*_kept for visualization, even if the UTM distance filter
    # removed some matches before UTM RANSAC.
    ransac_inliers_utm = None
    if ransac_inliers_utm_filtered is not None:
        ransac_inliers_utm = np.zeros(len(mkpts0_kept), dtype=bool)
        ransac_inliers_utm[utm_distance_keep_mask] = ransac_inliers_utm_filtered

    if H_utm is not None:
        H = H_utm
        ransac_inliers = ransac_inliers_utm
    else:
        H = H_pixel
        ransac_inliers = ransac_inliers_pixel

    tx_utm, ty_utm, theta_utm, theta_utm_deg = extract_euclidean_homography_params(H_utm)
    tx_utm_local, ty_utm_local, theta_utm_local, theta_utm_local_deg = extract_euclidean_homography_params(H_utm_local)

    num_ransac_inliers = 0 if ransac_inliers is None else int(np.sum(ransac_inliers))

    return RegistrationResult(
        mode=mode,
        image_source=image_source,

        img0=img0,
        img1=img1,
        mask0=mask0,
        mask1=mask1,

        mkpts0=mkpts0,
        mkpts1=mkpts1,
        mconf=mconf,
        keep_mask=keep_mask,

        mkpts0_kept=mkpts0_kept,
        mkpts1_kept=mkpts1_kept,
        mconf_kept=mconf_kept,

        apply_utm_distance_filter=apply_utm_distance_filter,
        utm_distance_threshold=utm_distance_threshold,
        utm_match_distances=utm_match_distances,
        utm_distance_keep_mask=utm_distance_keep_mask,
        mkpts0_utm_ransac=mkpts0_utm_ransac,
        mkpts1_utm_ransac=mkpts1_utm_ransac,

        mkpts0_utm=mkpts0_utm,
        mkpts1_utm=mkpts1_utm,
        mkpts0_utm_local=mkpts0_utm_local,
        mkpts1_utm_local=mkpts1_utm_local,

        H=H,
        H_pixel=H_pixel,
        H_utm=H_utm,
        H_utm_local=H_utm_local,

        tx_utm=tx_utm,
        ty_utm=ty_utm,
        theta_utm=theta_utm,
        theta_utm_deg=theta_utm_deg,

        tx_utm_local=tx_utm_local,
        ty_utm_local=ty_utm_local,
        theta_utm_local=theta_utm_local,
        theta_utm_local_deg=theta_utm_local_deg,

        ransac_inliers=ransac_inliers,
        ransac_inliers_pixel=ransac_inliers_pixel,
        ransac_inliers_utm=ransac_inliers_utm,

        utm_offset0=utm_offset0,
        utm_offset1=utm_offset1,

        num_matches=len(mkpts0),
        num_matches_after_mask=len(mkpts0_kept),
        num_matches_after_utm_distance_filter=len(mkpts0_utm_ransac),
        num_utm_distance_rejected=num_utm_distance_rejected,
        num_ransac_inliers=num_ransac_inliers,
    )


## Per-pair processing function

Runs section preparation, PhysDNet inference, mask building, the matching
sweep, and all plot generation for a single (ts1, ts2) pair.


In [ ]:
def make_matcher_for_spec(spec: SweepSpec):
    backend = spec.backend
    kwargs = spec.matcher_kwargs

    if backend == "minima_sp_lg":
        return MinimaMatcher(method="sp_lg", ckpt=MINIMA_CKPT, use_path=False)

    if backend in ("lightglue_superpoint", "lightglue_sift"):
        feature_type = "superpoint" if backend == "lightglue_superpoint" else "sift"
        return LightGlueFeatureMatcher(
            feature_type=feature_type,
            device=device,
            max_num_keypoints=kwargs.get("max_num_keypoints", LIGHTGLUE_MAX_NUM_KEYPOINTS),
            resize=kwargs.get("resize", LIGHTGLUE_RESIZE),
            depth_confidence=kwargs.get("depth_confidence", LIGHTGLUE_DEPTH_CONFIDENCE),
            width_confidence=kwargs.get("width_confidence", LIGHTGLUE_WIDTH_CONFIDENCE),
            filter_threshold=kwargs.get("filter_threshold", LIGHTGLUE_FILTER_THRESHOLD),
            mp=kwargs.get("mp", LIGHTGLUE_MP),
            flash=kwargs.get("flash", LIGHTGLUE_FLASH),
            compile_matcher=kwargs.get("compile_matcher", LIGHTGLUE_COMPILE),
        )

    if backend == "sift":
        return SIFTMatcher(
            nfeatures=kwargs.get("nfeatures", SIFT_NFEATURES),
            n_octave_layers=kwargs.get("n_octave_layers", SIFT_N_OCTAVE_LAYERS),
            contrast_threshold=kwargs.get("contrast_threshold", SIFT_CONTRAST_THRESHOLD),
            edge_threshold=kwargs.get("edge_threshold", SIFT_EDGE_THRESHOLD),
            sigma=kwargs.get("sigma", SIFT_SIGMA),
            matcher_type=kwargs.get("matcher_type", SIFT_MATCHER),
            ratio_test=kwargs.get("ratio_test", SIFT_RATIO_TEST),
            cross_check=kwargs.get("cross_check", SIFT_CROSS_CHECK),
            flann_trees=kwargs.get("flann_trees", SIFT_FLANN_TREES),
            flann_checks=kwargs.get("flann_checks", SIFT_FLANN_CHECKS),
            enforce_unique_matches=kwargs.get("enforce_unique_matches", SIFT_ENFORCE_UNIQUE_MATCHES),
            max_matches=kwargs.get("max_matches", SIFT_MAX_MATCHES),
        )

    raise ValueError(f"Unknown backend: {backend!r}")


def process_pair(ts1: int, ts2: int, label: Optional[str] = None):
    """
    Run the entire pipeline for one timestamp pair and save report plots.
    Intermediates (sections, inference, masks) are written under
    ``<report_dir>/intermediates/`` so one folder holds everything for one pair.
    Returns the report directory path, or None if the pair was skipped.
    """
    tag = f"{label}_{ts1}_{ts2}" if label else f"{ts1}_{ts2}"
    print(f"\n{'=' * 70}\nProcessing pair {tag}\n{'=' * 70}")
    t_start = time.perf_counter()

    # Resolve which group each timestamp belongs to.
    t1_group_idx = next((i for i, g in enumerate(groups_indices) if ts1 in g), None)
    t2_group_idx = next((i for i, g in enumerate(groups_indices) if ts2 in g), None)

    if t1_group_idx is None or t2_group_idx is None:
        print(f"  SKIP: ts1_valid={t1_group_idx is not None}, ts2_valid={t2_group_idx is not None} "
              "(at least one timestamp is in a turn segment).")
        return None

    lower1, upper1 = get_range_window(groups_indices[t1_group_idx], ts1)
    lower2, upper2 = get_range_window(groups_indices[t2_group_idx], ts2)
    print(f"  Section 1: {lower1}-{upper1}  Section 2: {lower2}-{upper2}")

    report_dir = make_report_dir(output_dir, ts1, ts2, label=label)
    intermediates = report_dir / "intermediates"
    intermediates.mkdir(parents=True, exist_ok=True)
    print(f"  Report dir:    {report_dir}")
    print(f"  Intermediates: {intermediates}")

    # Section preparation (saved under intermediates/section_0X/).
    section1 = prepare_data_range(
        input_dir=xtf_dir, xtf_file=xtf_file,
        segment_size=segment_size, upper_limit=upper_limit, side="both",
        data_lower_limit=lower1, data_upper_limit=upper1,
    )
    section2 = prepare_data_range(
        input_dir=xtf_dir, xtf_file=xtf_file,
        segment_size=segment_size, upper_limit=upper_limit, side="both",
        data_lower_limit=lower2, data_upper_limit=upper2,
    )
    save_sonar_section(section1, output_dir=str(intermediates / "section_01"), xtf_file=xtf_file)
    save_sonar_section(section2, output_dir=str(intermediates / "section_02"), xtf_file=xtf_file)

    # PhysDNet inference.
    inference1 = run_inference(section=section1, weight_path=weight_path, device=device)
    inference2 = run_inference(section=section2, weight_path=weight_path, device=device)
    save_inference_result(inference1, str(intermediates / "section_01" / "inference"))
    save_inference_result(inference2, str(intermediates / "section_02" / "inference"))

    # Terrain masks.
    terrain1 = terrain_mask(section=section1, inference=inference1, k=TERRAIN_MASK_K)
    terrain2 = terrain_mask(section=section2, inference=inference2, k=TERRAIN_MASK_K)
    save_terrain_result(terrain1, str(intermediates / "section_01"))
    save_terrain_result(terrain2, str(intermediates / "section_02"))

    # Final masks (closes over global FLIP_LEFT_FINAL_MASK and MASK_COMPONENTS).
    build_final_masks(
        section=section1, terrain=terrain1,
        output_mask_dir=str(intermediates / "section_01" / "masks"),
        lower=lower1, upper=upper1, title=f"Final masks {tag} sec1",
    )
    build_final_masks(
        section=section2, terrain=terrain2,
        output_mask_dir=str(intermediates / "section_02" / "masks"),
        lower=lower2, upper=upper2, title=f"Final masks {tag} sec2",
    )

    pings1 = data_load[lower1:upper1]
    pings2 = data_load[lower2:upper2]

    # Sweep closure - holds the per-pair data objects.
    def run_registration_for_spec(matcher, image_source):
        return run_parametrized_minima_registration(
            matcher=matcher, mode=MATCH_MODE,
            inference0=inference1, inference1=inference2,
            terrain0=terrain1, terrain1=terrain2,
            pings0=pings1, pings1=pings2,
            image_source=image_source,
            section0=section1, section1=section2,
            apply_mask=APPLY_MASK,
            ransac_reproj_threshold_pixel=RANSAC_REPROJ_THRESHOLD_PIXEL,
            ransac_reproj_threshold_utm=RANSAC_REPROJ_THRESHOLD_UTM,
            ransac_max_iters=RANSAC_MAX_ITERS,
            ransac_confidence=RANSAC_CONFIDENCE,
            ransac_outlier_percent=RANSAC_OUTLIER_PERCENT,
            flip_left_image=FLIP_LEFT_INPUT_IMAGE,
            apply_utm_distance_filter=APPLY_UTM_DISTANCE_FILTER,
            utm_distance_threshold=UTM_DISTANCE_THRESHOLD_METERS,
        )

    runs = run_sweep(SWEEP_SPECS, make_matcher_for_spec, run_registration_for_spec)

    plot_filtering_funnel(runs, report_dir)
    plot_utm_residuals(runs, report_dir)
    plot_homography_params(runs, report_dir)
    plot_summary_table(runs, report_dir)
    for r in select_top_runs(runs):
        plot_warp_diff(r, report_dir)

    elapsed = time.perf_counter() - t_start
    print(f"\n  Pair {tag} done in {elapsed:.1f}s")
    return report_dir


## Run all pairs


In [ ]:
results = {}
pair_folder_order: list[str] = []

for entry in TIMESTAMP_PAIRS:
    if len(entry) == 2:
        ts1, ts2 = entry
        label = None
    else:
        ts1, ts2, label = entry

    key = (ts1, ts2, label)
    try:
        path = process_pair(ts1, ts2, label=label)
    except Exception as exc:
        print(f"\nPair {key} FAILED: {type(exc).__name__}: {exc}")
        path = None
    results[key] = path
    if path is not None:
        pair_folder_order.append(path.name)

print("\n" + "=" * 70)
print("Per-pair summary")
print("=" * 70)
for (ts1, ts2, label), path in results.items():
    tag = f"{label}_{ts1}_{ts2}" if label else f"{ts1}_{ts2}"
    print(f"  {tag}: {str(path) if path else 'FAILED / SKIPPED'}")


## Cross-pair report

Aggregates every per-pair `summary.json` under `output_simplified/report/` into
a single CSV and a few line plots (median residual / inlier count vs pair).
Pairs appear left-to-right in the order they were processed above.


In [ ]:
report_root = Path(output_dir) / "report"
print(f"Aggregating summaries under {report_root}")

cross_paths = generate_cross_pair_report(
    report_root=str(report_root),
    out_dir=str(report_root),                # write CSV + plots into the report root
    pair_order=pair_folder_order or None,    # preserve TIMESTAMP_PAIRS ordering
)

print("\nSaved cross-pair artefacts:")
for k, v in cross_paths.items():
    print(f"  {k}: {v}")
